
# Supply Chain Delay Prediction

## Business Problem
Late deliveries increase operational costs, reduce customer satisfaction, and disrupt inventory planning.

### Objective
Build a machine learning model to predict whether an order will be delivered late using historical supply chain data and identify the key drivers of delivery delays.

### Dataset
- Records: ~180,000 orders
- Target Variable: Delivery Status
- Business Domain: Supply Chain & Logistics


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

pd.set_option('display.max_columns', None)


## Load Dataset

In [ ]:

df = pd.read_excel('Supply_Chain_Disruptions_Dataset_ML(2).xlsx')
df.head()


## Data Understanding

In [ ]:

print('Shape:', df.shape)
display(df.info())
display(df.describe(include='all'))
display(df.isnull().sum())


## Data Cleaning

In [ ]:

df = df.loc[:, ~df.columns.duplicated()]

if 'Benefit per order.1' in df.columns:
    df.drop(columns=['Benefit per order.1'], inplace=True)

df.head()


## Exploratory Data Analysis

In [ ]:

plt.figure(figsize=(8,5))
sns.countplot(x='Delivery Status', data=df)
plt.xticks(rotation=45)
plt.title('Delivery Status Distribution')
plt.show()


In [ ]:

plt.figure(figsize=(8,5))
sns.countplot(x='Shipping Mode', hue='Delivery Status', data=df)
plt.xticks(rotation=45)
plt.title('Shipping Mode vs Delivery Status')
plt.show()


In [ ]:

numeric_cols = df.select_dtypes(include=np.number).columns

plt.figure(figsize=(10,6))
df[numeric_cols].hist(figsize=(12,8))
plt.tight_layout()
plt.show()


## Feature Engineering

In [ ]:

df['Delayed'] = (df['Delivery Status'] == 'Late delivery').astype(int)

freq_cols = ['Order Region','Category Name']

for col in freq_cols:
    freq = df[col].value_counts()
    df[col] = df[col].map(freq)

label_cols = ['Shipping Mode','Market','Customer Segment']

for col in label_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

df.drop(columns=['Delivery Status'], inplace=True)

df.head()


In [ ]:

corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


## Train-Test Split

In [ ]:

X = df.drop('Delayed', axis=1)
y = df['Delayed']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)


## Model Building

In [ ]:

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Naive Bayes': GaussianNB()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)

    results.append({
        'Model': name,
        'Accuracy': accuracy
    })

results_df = pd.DataFrame(results)
results_df.sort_values('Accuracy', ascending=False)


## Hyperparameter Tuning - Decision Tree

In [ ]:

params = {
    'max_depth':[5,10,15],
    'min_samples_split':[2,5,10]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    params,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)


## Feature Importance

In [ ]:

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

importance.head(10)


In [ ]:

plt.figure(figsize=(8,5))
sns.barplot(
    data=importance.head(10),
    x='Importance',
    y='Feature'
)
plt.title('Top 10 Important Features')
plt.show()



# Business Insights

1. Identify high-risk orders before dispatch.
2. Improve logistics planning using key predictors.
3. Optimize shipping modes associated with delays.
4. Reduce customer dissatisfaction and operational losses.

# Conclusion

This project developed multiple machine learning models to predict late deliveries using supply chain data. Model performance was evaluated using classification metrics, and feature importance analysis was used to identify operational drivers of delays.
